In [1]:
# Etape 1: tokenisation
# ici nous n'allons pas entré dans le detail de l'algorithme de tokenisation Byte-level "Byte-Pair Encoding" ou "BPE"
# L'agorithme sera l'object d'une etude plus tard

import tiktoken

MODEL: str = "gpt2"
INPUT: str = "hello world comment allez vous?"

enc = tiktoken.encoding_for_model(MODEL)
token_list: list[int] = enc.encode(INPUT)



In [2]:
# Ci dessous le resultat de la tokenisation de la variable INPUT. 
for token in token_list:
    print(f"'{enc.decode([token])}' -> value {token}")

'hello' -> value 31373
' world' -> value 995
' comment' -> value 2912
' alle' -> value 28654
'z' -> value 89
' v' -> value 410
'ous' -> value 516
'?' -> value 30


In [3]:
# Chaque token à sa correspondance dans la table d'embeding sous la forme d'un vecteur 
# de taille n_embd . Parametre visible dans le fichier "config.json".
# Dans le cas de notre model gpt2, chaque token est representé par un vecteur de 768 dimentions.

# Commencon par ouvrir notre fichier safetensor et recuperer notre matrice d'embedding.
# La taille du vocabulaire etant 50257, la matrice sera de dimention We = 50257 * 768


from safetensors import safe_open
import numpy as np

MODE_DISPLAY: bool = False
TENSOR_PATH: str = "/mnt/data/gpt2/model.safetensors"
TENSOR_NAME_EMBEDING: str = 'wte.weight'
TENSOR_NAME_POSITIONAL_EMBEDING: str = 'wpe.weight'

tensor_embedding : np.ndarray
tensor_positional_embedding: np.ndarray
try:
    with safe_open(TENSOR_PATH, framework='np') as f:
        if MODE_DISPLAY:
            for key in f.keys():
                print(f"Tensor name: {key}")
            print(f"Un tensor est de type: {type(f.get_tensor(TENSOR_NAME_EMBEDING))}")
        tensor_embedding = f.get_tensor(TENSOR_NAME_EMBEDING)
        tensor_positional_embedding = f.get_tensor(TENSOR_NAME_POSITIONAL_EMBEDING)
        print(f"La taille de la matrice d'embeding est: {len(tensor_embedding)} * {len(tensor_embedding[0])}")
        print(f"La taille de la matrice d'embeding positionelle est: {len(tensor_positional_embedding)} * {len(tensor_positional_embedding[0])}")
except Exception as e:
    print(f"Erreur dans la tentative d'ouvir {e}")


La taille de la matrice d'embeding est: 50257 * 768
La taille de la matrice d'embeding positionelle est: 1024 * 768


In [ ]:
# Construire la matrice d'embeddings de notre séquence en récupérant, pour chaque token,
# son vecteur d'embedding dans la matrice des poids préentraînés de GPT-2 (wte.weigth) ,
# en utilisant son identifiant comme indice de ligne.

tokens_embedding = [tensor_embedding[token] for token in token_list]
assert(len(tokens_embedding) == len(token_list))
assert(len(tokens_embedding[0]) == 768)

# Nous contruissont aussi la matrice d'embedding positionel (wpe.weight).
# celle ci sera additionner a notre matrice d'embeddings 
# permettant d'encoder l'information positionelle de notre token dans son vecteur d'embedding.
# La taille de wpe.weight et de 1024 * 768. 1024 etant la fenetre de context maximum du mdoel gpt2.

token_positional_embedding = [tensor_positional_embedding[idx] for idx, token in enumerate(token_list,0)]
assert(len(token_positional_embedding) == len(tokens_embedding) and len(token_positional_embedding[0]) == len(tokens_embedding[0]))

#combined_embeddings is the addition of tokens_embedding and token_positional_embedding
combined_embeddings = np.add(tokens_embedding, token_positional_embedding)
print(f"tokens_embedding[0][0] = {tokens_embedding[0][0]}")
print(f"token_positional_embedding[0][0] = {token_positional_embedding[0][0]}")
print(f"tokens_embedding[0][0] + token_positional_embedding[0][0] = {combined_embeddings[0][0]}")
assert(combined_embeddings[1][1] == (tokens_embedding[1][1] + token_positional_embedding[1][1]))

tokens_embedding[0][0] = -0.0818910300731659
token_positional_embedding[0][0] = -0.01882071979343891
tokens_embedding[0][0] + token_positional_embedding[0][0] = -0.10071174800395966
